# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 476ms
Checked 79 packages in 12ms


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
from dotenv import load_dotenv
import os
import json
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr


c:\Users\brade\OneDrive - University of Cincinnati\Intro to Agentic AI\Module 7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")


Azure model configured for the OpenAI Agents SDK.


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
persona = (
    "Coach Rowan, a blunt, no-nonsense fitness trainer who keeps feedback short and direct, "
    "but always tells you exactly what to fix. Coach Rowan asks one question at a time, "
    "gives credit only when an answer is actually correct, and never pads a correction "
    "with fluff."
)

reader = PdfReader("notes/coach.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text + "\n"

print(persona)
print(notes[:300], "...")


Coach Rowan, a blunt, no-nonsense fitness trainer who keeps feedback short and direct, but always tells you exactly what to fix. Coach Rowan asks one question at a time, gives credit only when an answer is actually correct, and never pads a correction with fluff.
IT3025 Assignment 5: Coach Rowan Fitness and Nutrition Notes 
Study notes - macros, calories, training basics, muscles, recovery 
 
Macronutrients 
The three macronutrients are protein, carbohydrates, and fat. Protein provides 4 calories 
per gram, 
carbohydrates provide 4 calories per gram, and fat ...


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
LOG_PATH = "missed_questions.txt"

@function_tool
def log_missed_question_tool(question: str) -> str:
    """Record a quiz question the student got wrong or didn't know, so they can review it later.

    Args:
        question: The exact quiz question the student missed.
    """
    print(f"Tool called to log a missed question: {question}")
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print("Tool ready:", log_missed_question_tool.name)


Tool ready: log_missed_question_tool


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
@function_tool
def fetch_bonus_question_tool() -> str:
    """Fetch a fun bonus sports trivia question, for when the student asks for a bonus question or a break from the notes."""
    print("Tool called to fetch a bonus sports question")
    try:
        r = requests.get("https://opentdb.com/api.php",
                         params={"amount": 1, "type": "multiple", "category": 21}, timeout=20)
        data = r.json()
    except (requests.exceptions.RequestException, ValueError):
        return "The trivia API is temporarily unavailable - skip the bonus question and continue the quiz."
    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - skip it and continue the quiz."
    q = data["results"][0]
    return f"Bonus question: {html.unescape(q['question'])} (correct answer: {html.unescape(q['correct_answer'])})"

print("Tool ready:", fetch_bonus_question_tool.name)


Tool ready: fetch_bonus_question_tool


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
SCORE_PATH = "score.json"

@function_tool
def track_score_tool(correct: bool) -> str:
    """Record whether the student's latest quiz answer was right or wrong, and get their running score.

    Args:
        correct: True if the student answered correctly, False otherwise.
    """
    print(f"Tool called to track score (correct={correct})")
    try:
        with open(SCORE_PATH, "r", encoding="utf-8") as f:
            score = json.load(f)
    except (FileNotFoundError, ValueError):
        score = {"right": 0, "wrong": 0}
    score["right" if correct else "wrong"] += 1
    with open(SCORE_PATH, "w", encoding="utf-8") as f:
        json.dump(score, f)
    return f"Score so far: {score['right']} right, {score['wrong']} wrong."

print("Tool ready:", track_score_tool.name)


Tool ready: track_score_tool


## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
instructions = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the study notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

After every answer the student gives to a quiz question, use track_score_tool to record whether
it was correct.

If the student's answer was wrong, or they say "I don't know", you must also record it as a missed
question using log_missed_question_tool before moving on to the next question.

If the student asks for a bonus question, or seems to want a break from the notes, use
fetch_bonus_question_tool to get one. Ask the bonus question, wait for their answer, and only then
reveal the correct answer. Bonus questions do not need to relate to the notes, and answers to them
should not be scored or logged.

IMPORTANT: apart from bonus questions, only ask about things that are actually covered in the notes
above. If the student asks about something outside the notes, say that it isn't covered in this
study set.
"""

coach_rowan = Agent(
    name="Coach Rowan",
    instructions=instructions,
    tools=[log_missed_question_tool, fetch_bonus_question_tool, track_score_tool],
    model=azure_model,
)

print("Agent built:", coach_rowan.name)


Agent built: Coach Rowan


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [11]:
result = await Runner.run(coach_rowan, [
    {"role": "user", "content": "Quiz me with one question."},
    {"role": "assistant", "content": "Question 1: Which macronutrient has 9 calories per gram?"},
    {"role": "user", "content": "Carbohydrates."},
])
print(result.final_output)

# Confirm the tools actually did something on disk
for path in ["missed_questions.txt", "score.json"]:
    try:
        with open(path, encoding="utf-8") as f:
            print(path, "->", f.read())
    except FileNotFoundError:
        print(path, "not created yet")


Tool called to track score (correct=False)
Tool called to log a missed question: Which macronutrient has 9 calories per gram?
Wrong. Fat has 9 calories per gram; carbs have 4.

Next question: What is the main nutrient used to repair and build muscle tissue?
missed_questions.txt -> Which macronutrient has 9 calories per gram?
Which macronutrient has 9 calories per gram?

score.json -> {"right": 0, "wrong": 2}


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [12]:
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(coach_rowan, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called to track score (correct=False)Tool called to log a missed question: How many calories per gram does fat provide?

Tool called to track score (correct=False)Tool called to log a missed question: What are the three macronutrients?

Tool called to fetch a bonus sports question


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?

   I copied over the persona, the notes loading, the tool function bodie, and the wording from the system prompt. The async client and model wrapper needed a change, with chat() becoming async. 

2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?

   Runner.run is doing the loop for me. It calls the model, sees which tool I requested, runs the matching function, feeds the result back, and repeats this until there is an answer. The SDK matches by name using the tools list, which is what if/elif did before.

3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?

   The wording was carried over directly. The instructions are the same text as a system prompt. The one requirment is that tool names in my rules match the function names. 

4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

   I would write one typed function with a docstring, the @function_tool decorator, adding it to tools, and a rule sentence in the intructions. I would do this instead of writing a JSON schema, an elif branch in a dispatcher, or any chnage in the loop. 
   
## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.